In [0]:
CATALOG = "my_living_index"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Real Income vs CPI
print("Building gold_real_income_vs_cpi...")

# Annual average CPI
cpi_annual = (
    spark.table(f"{CATALOG}.silver.silver_cpi_monthly")
    .filter(F.col("division") == "overall")
    .groupBy("year")
    .agg(F.avg("index").alias("cpi_index_avg"))
    .orderBy("year")
)

# YoY inflation from annual CPI
cpi_with_yoy = (
    cpi_annual
    .withColumn("cpi_lag", F.lag("cpi_index_avg", 1).over(Window.orderBy("year")))
    .withColumn("yoy_inflation_pct", F.round(((F.col("cpi_index_avg") - F.col("cpi_lag")) / F.col("cpi_lag")) * 100, 2))
    .drop("cpi_lag")
)

# Household income
hh_income = (
    spark.table(f"{CATALOG}.silver.silver_hh_income_national")
    .select("year", "income_mean", "income_median", "mean_growth_pct", "median_growth_pct")
)

# Join income to CPI on year
gold_real_income = (
    hh_income
    .join(cpi_with_yoy, on="year", how="inner")
    # Deflate nominal to real income
    .withColumn("real_income_mean", F.round(F.col("income_mean") / (F.col("cpi_index_avg") / 100), 2))
    .withColumn("real_income_median", F.round(F.col("income_median") / (F.col("cpi_index_avg") / 100), 2))
    # YoY real income growth
    .withColumn("real_mean_lag", F.lag("real_income_mean", 1).over(Window.orderBy("year")))
    .withColumn("real_income_growth_pct", F.round(((F.col("real_income_mean") - F.col("real_mean_lag")) / F.col("real_mean_lag")) * 100, 2))
    .drop("real_mean_lag")
    .withColumn("income_beats_inflation", F.when(F.col("mean_growth_pct") > F.col("yoy_inflation_pct"), True).otherwise(False))
    .withColumn("real_growth_gap", F.round(F.col("mean_growth_pct") - F.col("yoy_inflation_pct"), 2))
    .orderBy("year")
)

gold_real_income.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.gold_real_income_vs_cpi")

print(f"gold_real_income_vs_cpi: {gold_real_income.count():,} rows")

# Affordability by State
print("\nBuilding gold_affordability_by_state...")

gold_afford = (
    spark.table(f"{CATALOG}.silver.silver_hies_state")
    .select("year", "state", "income_mean", "income_median", "expenditure_mean", "expenditure_to_income_ratio", "monthly_surplus", "affordability_rank", "gini", "poverty")
    # Label affordability tier based on expenditure ratio
    .withColumn("affordability_tier",
                F.when(F.col("expenditure_to_income_ratio") < 0.60, "Comfortable")
                 .when(F.col("expenditure_to_income_ratio") < 0.75, "Moderate")
                 .when(F.col("expenditure_to_income_ratio") < 0.90, "Stretched")
                 .otherwise("Critical"))
    # Flag states where surplus is less than RM500/month
    .withColumn("low_surplus_flag", F.col("monthly_surplus") < 500)
    .orderBy("affordability_rank")
)

gold_afford.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.gold_affordability_by_state")

print(f"gold_affordability_by_state: {gold_afford.count():,} rows")

# Income Inequality Snapshot
print("\nBuilding gold_income_inequality...")

gold_inequality = (
    spark.table(f"{CATALOG}.silver.silver_income_percentile")
    .groupBy("year", "income_group")
    .agg(
        F.avg("income_mean").alias("avg_income"),
        F.avg("income_median").alias("avg_income_median"),
        F.avg("income_minimum").alias("floor_income"),
        F.avg("income_maximum").alias("ceiling_income"),
        F.avg("income_range").alias("avg_income_range"),
        F.avg("percentile").alias("percentile_count"),
    )
    .orderBy("year", "income_group")
)

# Compute T20-to-B40 ratio
b40 = gold_inequality.filter(F.col("income_group") == "B40").select("year", F.col("avg_income").alias("b40_avg"))

t20 = gold_inequality.filter(F.col("income_group") == "T20").select("year", F.col("avg_income").alias("t20_avg"))

ratio = b40.join(t20, on="year").withColumn("t20_to_b40_ratio", F.round(F.col("t20_avg") / F.col("b40_avg"), 2))

gold_inequality_final = gold_inequality.join(ratio.select("year", "t20_to_b40_ratio"), on="year", how="left")

gold_inequality_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.gold_income_inequality")

print(f"gold_income_inequality: {gold_inequality_final.count():,} rows")

# Final summary report
print("\nBuilding gold_cost_of_living_report...")

gold_report = (
    spark.table(f"{CATALOG}.gold.gold_real_income_vs_cpi")
    .select("year", "income_mean", "income_median", "real_income_mean", "real_income_median", "cpi_index_avg", "yoy_inflation_pct", "mean_growth_pct", "real_income_growth_pct", "income_beats_inflation", "real_growth_gap")
    .withColumn("purchasing_power_status", 
                F.when(F.col("real_income_growth_pct") > 2, "IMPROVING")
                 .when(F.col("real_income_growth_pct") > 0, "MARGINALLY IMPROVING")
                 .when(F.col("real_income_growth_pct").isNull(), "BASELINE / NO PRIOR DATA")
                 .otherwise("DECLINING"))
    # Narrative label for each period
    .withColumn("period_verdict",
                F.when(F.col("income_beats_inflation") == True, "Income outpaced inflation")
                 .when(F.col("income_beats_inflation") == False, "Income eroded purchasing power")
                 .otherwise("Insufficient data"))
    .orderBy("year")
)

gold_report.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.gold_cost_of_living_report")

print(f"gold_cost_of_living_report: {gold_report.count():,} rows")
display(gold_report)

print("\nAll Gold tables done.")